# Importación de librerías

In [306]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import pandas as pd
import numpy as np

# Datos

In [307]:
df = pd.read_parquet("data/minable_view_post.parquet")

# Predecir 0 o >0

In [308]:
features = [
    col
    for col in df.columns.to_list()
    if col
    not in [
        "id",
        "track_name",
        "artist_name",
        "new_popularity_2025",
        "genres",
        "main_genre",
        "subgenre_1",
        "subgenre_2",
        "genres_artists",
        "acousticness_cat",
        "danceability_cat",
        "energy_cat",
        "explicit_cat",
        "instrumentalness_cat",
        "liveness_cat",
        "loudness_cat",
        "speechiness_cat",
        "tempo_cat",
        "valence_cat",
        "month_cat",
        "featuring_cat",
        "popularity_2021_cat",
        "duration_cat",
        "key_cat",
        "imputed_date_cat",
        "mode_cat",
        "year_cat",
        "genres_from_artists_cat",
        "popularity_2025_cat",
        "album",
        "popularity_2024",
        "popularity_2024_cat",
        "popularity_2021",
    ]
]
target = "new_popularity_2025"

In [309]:
X = df[features]
y = df[target] > 0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# FINAL 0 O > 0

In [ ]:
numeric_features = [
    col for col in X_train.columns if X_train[col].dtype in ["int64", "float64"]
]
categorical_features = [
    col for col in X_train.columns if X_train[col].dtype == "object"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("clf", RandomForestClassifier())
    ]
)

param_grid = [
    {
        "clf": [RandomForestClassifier(
            class_weight='balanced'
        )],
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_split": [2, 5],
        "clf__min_samples_leaf": [1, 2],
        "clf__max_features": ["sqrt", "log2"],
    },
    {
        "clf": [GradientBoostingClassifier()],
        "clf__n_estimators": [100, 200, 300],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__max_depth": [3, 5, 10],
        "clf__subsample": [0.8, 1.0],
        "clf__max_features": ["sqrt", "log2"],
    },
    {
        "clf": [LogisticRegression(max_iter=1000, solver="liblinear", class_weight='balacecd')],
        "clf__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "clf__penalty": ["l1", "l2"],
    },
    {
        "clf": [XGBClassifier(eval_metric="logloss")],
        "clf__n_estimators": [100, 200, 300],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__max_depth": [3, 5, 10],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__gamma": [0, 1, 5],
    },
]

grid = GridSearchCV(pipeline, param_grid, scoring='f1_macro', cv=5, verbose=2, n_jobs=-1)

grid.fit(X_train, y_train)

best_model_f1_macro = grid.best_estimator_

Fitting 5 folds for each of 596 candidates, totalling 2980 fits


[CV] END clf=RandomForestClassifier(), clf__class_weight=None, clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time= 6.0min
[CV] END clf=RandomForestClassifier(), clf__class_weight=None, clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time= 6.1min
[CV] END clf=RandomForestClassifier(), clf__class_weight=None, clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time= 6.1min
[CV] END clf=RandomForestClassifier(), clf__class_weight=None, clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time= 6.1min
[CV] END clf=RandomForestClassifier(), clf__class_weight=None, clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time= 6.

In [285]:
best_model_f1_macro

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['acousticness',
                                                   'danceability',
                                                   'duration_ms', 'energy',
                                                   'explicit',
                                                   'instrumentalness', 'key',
                                                   'liveness', 'loudness',
                                                   'mode', 'speechiness',
                                                   'tempo', 'valence', 'year',
                                                   'month_sin', 'month_cos',
                                                   'dayofweek_sin',
                                                   'dayofweek_cos',
                                                   'years_since_release',
                                                   'energy_valence',
                                                   'speechiness_explicit',
                                                   'danceability_tempo']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['season'])])),
                ('clf',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        min_samples_leaf=2,
                                        n_estimators=300))])

In [287]:
y_pred = best_model_f1_macro.predict(X_test)
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

       False       0.56      0.64      0.60     19439
        True       0.79      0.73      0.76     36652

    accuracy                           0.70     56091
   macro avg       0.68      0.69      0.68     56091
weighted avg       0.71      0.70      0.70     56091



## Prueba de threshold

In [292]:
y_proba = best_model_f1_macro.predict_proba(X_test)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"\n--- Umbral {t:.2f} ---")
    print(f"F1_macro: {f1:.4f}")
    print(classification_report(y_test, y_pred))


--- Umbral 0.30 ---
F1_macro: 0.5350
              precision    recall  f1-score   support

       False       0.88      0.15      0.26     19439
        True       0.69      0.99      0.81     36652

    accuracy                           0.70     56091
   macro avg       0.78      0.57      0.53     56091
weighted avg       0.75      0.70      0.62     56091


--- Umbral 0.40 ---
F1_macro: 0.6389
              precision    recall  f1-score   support

       False       0.67      0.36      0.47     19439
        True       0.73      0.90      0.81     36652

    accuracy                           0.72     56091
   macro avg       0.70      0.63      0.64     56091
weighted avg       0.71      0.72      0.69     56091


--- Umbral 0.50 ---
F1_macro: 0.6791
              precision    recall  f1-score   support

       False       0.56      0.64      0.60     19439
        True       0.79      0.73      0.76     36652

    accuracy                           0.70     56091
   macro avg  

In [320]:
# Filtra solo las columnas numéricas
num_cols = X_train.select_dtypes(include=["int64", "float64"])

# Ahora sí puedes calcular la matriz de correlaciones
corr_matrix = num_cols.corr().abs()

# Parte superior de la matriz
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Detecta colinealidades altas (> 0.9 por ejemplo)
high_corr_cols = [column for column in upper.columns if any(upper[column] > 0.9)]

# Opcional: elimina esas columnas del X_train
X_train_filtered = X_train.drop(columns=high_corr_cols)

In [323]:
X_train_filtered

,acousticness,danceability,duration_ms,energy,explicit,instrumentalness,key,liveness,loudness,mode,...,is_live,is_common_key,is_instrumental,is_energetic,is_duration_friendly,is_danceable,is_acoustic,energy_valence,speechiness_explicit,danceability_tempo
133022,0.159000,0.814,242776,0.5510,0,0.000002,0,0.1510,-8.599,1,...,False,True,False,False,False,True,False,0.228665,0.0000,94.447606
154070,0.601000,0.497,284146,0.7670,0,0.000000,10,0.1260,-3.729,1,...,False,False,False,True,False,False,False,0.245440,0.0000,65.608473
194694,0.245000,0.639,193060,0.5930,0,0.000000,4,0.2420,-6.055,1,...,False,False,False,False,True,True,False,0.418658,0.0000,100.261656
164012,0.000624,0.718,227786,0.7760,0,0.000005,7,0.2070,-5.208,0,...,False,True,False,True,True,True,False,0.482672,0.0000,86.170770
140410,0.206000,0.768,252333,0.5990,0,0.000462,0,0.0704,-9.582,1,...,False,True,False,False,False,True,False,0.476205,0.0000,101.995008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119879,0.388000,0.763,196026,0.5260,1,0.000005,10,0.1260,-11.046,0,...,False,False,False,False,True,True,False,0.132026,0.0598,90.041630
259178,0.102000,0.658,145116,0.8930,1,0.000000,10,0.2080,-6.558,0,...,False,False,False,True,True,True,False,0.243789,0.4020,74.138834
131932,0.935000,0.441,198666,0.0663,0,0.000051,7,0.2320,-17.778,1,...,False,True,False,False,True,False,True,0.029769,0.0000,38.953971
146867,0.257000,0.597,161929,0.5020,0,0.000000,4,0.1150,-5.726,1,...,False,False,False,False,True,False,False,0.283128,0.0000,59.045688


In [326]:
from scipy.stats import zscore

# Filtrar outliers si z-score absoluto > 3 en alguna variable numérica
z_scores = np.abs(zscore(X_train.select_dtypes(include=np.number)))
mask = (z_scores < 3).all(axis=1)
X_train_no_outliers = X_train_filtered[mask]
y_train_no_outliers = y_train[mask]

In [327]:
print("🔢 Tamaño original:", X_train_filtered.shape, y_train.shape)
print("📉 Tamaño sin outliers:", X_train_no_outliers.shape, y_train_no_outliers.shape)

🔢 Tamaño original: (224360, 33) (224360,)
📉 Tamaño sin outliers: (194087, 33) (194087,)


In [337]:
numeric_features = [
    col for col in X_train_no_outliers.columns if X_train_no_outliers[col].dtype in ["int64", "float64"]
]
categorical_features = [
    col for col in X_train_no_outliers.columns if X_train_no_outliers[col].dtype == "object"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", RandomForestClassifier())])

In [343]:
numeric_features = [
    col for col in X_train_no_outliers
    .columns if X_train_no_outliers[col].dtype in ["int64", "float64"]
]
categorical_features = [
    col for col in X_train_no_outliers.columns if X_train_no_outliers[col].dtype == "object"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

param_grid = [
    {
        "clf": [RandomForestClassifier(class_weight="balanced")],
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_split": [2, 5],
        "clf__min_samples_leaf": [1, 2],
        "clf__max_features": ["sqrt", "log2"],
    },
    {
        "clf": [GradientBoostingClassifier()],
        "clf__n_estimators": [100, 200, 300],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__max_depth": [3, 5, 10],
        "clf__subsample": [0.8, 1.0],
        "clf__max_features": ["sqrt", "log2"],
    },
    {
        "clf": [
            LogisticRegression(
                max_iter=1000, solver="liblinear", class_weight="balanced"
            )
        ],
        "clf__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "clf__penalty": ["l1", "l2"],
    },
    {
        "clf": [XGBClassifier(eval_metric="logloss")],
        "clf__n_estimators": [100, 200, 300],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__max_depth": [3, 5, 10],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__gamma": [0, 1, 5],
    },
]

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", RandomForestClassifier())])

grid = GridSearchCV(
    pipeline, param_grid, scoring="f1_macro", cv=2, verbose=2, n_jobs=-1
)

grid.fit(X_train, y_train)

best_model_f1_macro = grid.best_estimator_

Fitting 2 folds for each of 514 candidates, totalling 1028 fits
[CV] END clf=RandomForestClassifier(class_weight='balanced'), clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=5, clf__n_estimators=100; total time= 2.0min
[CV] END clf=RandomForestClassifier(class_weight='balanced'), clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=5, clf__n_estimators=100; total time= 2.1min
[CV] END clf=RandomForestClassifier(class_weight='balanced'), clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time= 2.1min
[CV] END clf=RandomForestClassifier(class_weight='balanced'), clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=100; total time= 2.1min
[CV] END clf=RandomForestClassifier(class_weight='balanced'), clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=2, clf_

In [344]:
best_model_f1_macro

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['acousticness',
                                                   'danceability',
                                                   'duration_ms', 'energy',
                                                   'explicit',
                                                   'instrumentalness', 'key',
                                                   'liveness', 'loudness',
                                                   'mode', 'speechiness',
                                                   'tempo', 'valence', 'year',
                                                   'month_sin', 'month_cos',
                                                   'dayofweek_sin',
                                                   'dayofweek_cos',
                                                   'energy_valence',
                                                   'speechiness_explicit',
                                                   'danceability_tempo']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['season'])])),
                ('clf',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        min_samples_leaf=2,
                                        n_estimators=300))])

In [352]:
y_pred = best_model_f1_macro.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.55      0.64      0.59     19439
        True       0.79      0.73      0.76     36652

    accuracy                           0.70     56091
   macro avg       0.67      0.68      0.68     56091
weighted avg       0.71      0.70      0.70     56091



In [350]:
X_train_pos = X_train[y_train]
y_train_pos = y_train[y_train]

In [356]:
import joblib

# Supongamos que tu modelo entrenado se llama 'modelo_final'
joblib.dump(best_model_f1_macro, "modelo_final.pkl")

['modelo_final.pkl']